# DADA-2000 T2 — **D1 / D2: why KIP-on is negative and where the gradient goes**

Plan: `.project/plans/katvad-kip-loss-scale-diagnosis.md` — **read §2 and §3 before changing a flag here.**
Record being explained: `outputs/v1/DADA2000_orig_phase4/` (3 seeds, paired, `kip.enabled` the only diff).

**What this notebook is.** Two *diagnostics*. It trains nothing, tunes nothing and writes no checkpoint.
Both bars below were written down before either measurement existed (**C33**: the condition lives in the
threshold, not in the prose next to it).

**The result being diagnosed** — T2 micro is the only interval excluding zero, and all 3 seeds agree:

| eval | metric | kip_off | kip_on | paired Δ | t95 (n=3) |
|---|---|---:|---:|---:|---|
| T2 in-domain | `auc_macro` | 0.6248 | 0.6046 | −0.0201 | [−0.0698, +0.0296] |
| T2 in-domain | `auc` micro | 0.6182 | 0.6054 | **−0.0129** | **[−0.0249, −0.0008]** |
| DoTA 0-shot | `auc_macro` | 0.6113 | 0.6071 | −0.0042 | [−0.1024, +0.0941] |
| DoTA 0-shot | `auc` micro | 0.5856 | 0.5914 | +0.0058 | [−0.0790, +0.0905] |

**The two questions.**

* **D1** — `kip_rec` ends at ≈11.6 and is **93 % of `total`**. Is that a bad fit, a good fit, or worse than a
  constant? A bare MSE against **unnormalized** flow statistics has a magnitude set by pixel units, not by
  the head. §2 measures what a constant predictor scores on the same `e_O`.
* **D2** — a 93 % *loss* share is not a 93 % *gradient* share. §3 measures what actually arrives at the
  shared temporal encoder, and whether the two gradients **conflict** or are merely unequal.

**Pre-registered bars** (plan §2 and §3):

| id | quantity | bar |
|---|---|---|
| **D1-a** | `R²_global = 1 − K/V` | ≥ 0.20 **PASS** · < 0.20 **STOP** |
| **D1-b** | `K` vs `W` (item-mean oracle) | `K < W` **PASS** · `K ≥ W` **REFUTES** the motion premise |
| **D1-c** | `V` | ≥ 4.0 **CONFIRMS** `lambda_rec=1.0` is an overweight; equalizing weight = `1/V` |
| **D1-d** | `predicted/Z` projection round-trip | ∈ [0.9, 1.1] else **HARD STOP** |
| **D2-a** | `rho = |g_KIP|/|g_task|` at trunk | ≥ 1.0 **CONFIRMS** capture · < 0.30 **REFUTES** the whole story |
| **D2-b** | `cos(g_kip_rec, g_task)` at trunk | ≤ −0.1 conflict · \|cos\| < 0.1 orthogonal · ≥ +0.1 aligned |

> **C24 stands whatever these return.** Every KIP-on arm on `main` is a fixed ~50 % channel shift — the gate
> MLP never receives a gradient and there is no `kip.gate_type` on this branch. Nothing here licenses the
> words "motion-gated".

> **Nothing is repaired in this notebook.** The four options (z-score `e_O` / lower `lambda_rec` / detach
> `v^t` / conclude) are gated behind the decision table in plan §4, which §4 below prints for you.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE = '/content/drive/MyDrive/Thesis'
os.environ['PROJECT_ROOT']       = DRIVE
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE}/outputs'

os.environ['REPO'] = f'{DRIVE}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'tools' / 'grad_probe.py').is_file(), (
    f'no checkout at {REPO}, or it predates the D2 probe -- push the branch that '
    'carries core/tools/grad_probe.py before running this notebook')
os.environ['PYTHONPATH'] = str(REPO)
sys.path.insert(0, str(REPO))

from core import constants  # noqa: E402

DATASET = constants.DADA_ORIGIN_DATASET
T2 = constants.DATA_ROOT / DATASET
CLIP_DIR = constants.CLIP_CACHE_DIR / DATASET
FLOW_DIR = constants.FLOW_CACHE_DIR / DATASET
KNN_CACHE = constants.KNN_CACHE_DIR / DATASET / constants.KNN_CACHE_FILENAME
RUNS = constants.OUTPUT_ROOT / f'{DATASET}_phase4'      # phase_4's arms, synced from the VM
DIAG = constants.OUTPUT_ROOT / f'{DATASET}_diag_kip_loss_scale'
DIAG.mkdir(parents=True, exist_ok=True)
P5 = Path('/content/p5')                                # VM-local scratch, never Drive
P5.mkdir(parents=True, exist_ok=True)

SEEDS = (2024, 2025, 2026)

# --- the pre-registered bars, as code. Changing a number here changes the claim. ---
D1A_R2_PASS = 0.20        # R^2 against the global-mean predictor
D1C_OVERWEIGHT_MSE = 4.0  # == core.eda.report.FLOW_TARGET_SCALE_WARN_MSE
D1D_RATIO_LO, D1D_RATIO_HI = 0.90, 1.10
D2A_CONFIRM, D2A_REFUTE = 1.00, 0.30
D2B_COS = 0.10

print(f'dataset {DATASET}')
for name, path in (('corpus', T2), ('clip', CLIP_DIR), ('flow', FLOW_DIR),
                   ('knn', KNN_CACHE), ('phase-4 runs', RUNS), ('diag out', DIAG)):
    print(f'  {name:13s} {path}   {"OK" if path.exists() else "MISSING"}')

In [ ]:
%%bash
# transformers is PINNED for the same reason phase_4 pins it: core/models/clip_text.py reaches
# into three transformers internals (lesson C7) and v5 flattened the nested .text_model. D2
# builds the REAL graph (--text-encoder clip) so it hits that path; D1 does not.
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- D2 will be slow but will run'
df -h /content | tail -1

### 0.1 Smoke-test the text tower — one second, no download

D2 runs `--text-encoder clip`, so it fails on the same v5 incompatibility phase_4 hit **~30 s in,
after the run dir already looked started**. Catch it here instead.

In [ ]:
import torch
import transformers
from transformers import CLIPTextConfig, CLIPTextModel

from core.models.clip_text import SoftPromptCLIPTextModel

print(f'python {sys.version.split()[0]} | torch {torch.__version__} | '
      f'transformers {transformers.__version__}')

try:
    tiny = CLIPTextModel(CLIPTextConfig(
        vocab_size=64, hidden_size=32, intermediate_size=64, num_hidden_layers=1,
        num_attention_heads=2, max_position_embeddings=77, eos_token_id=2))
    encoder = SoftPromptCLIPTextModel(clip_model=tiny, num_soft_prompts=4)
    ids = torch.tensor([[0, 5, 6, 2]])
    pooled = encoder(input_ids=ids, attention_mask=torch.ones_like(ids)).pooler_output
    print(f'text tower OK -- pooler_output {tuple(pooled.shape)}')
except AttributeError as exc:
    raise SystemExit(
        f'The text tower does not run on this stack: {exc}\n'
        f'  transformers {transformers.__version__} -- the pipeline needs 4.56.*.\n'
        '  Fix: run the cell above, then Runtime -> Restart session, then re-run from 0.\n'
        '  Do NOT port clip_text.py to fix this: it sits on the score path (C7, C14).') from exc

### 0.2 Stage the caches on VM-local disk

Same reason as phase_4 §0.2: D1 opens **two `.npy` per training item** across the whole train split and D2
builds datasets per probe run. Against a Drive FUSE mount that is the wall clock. Bit-identical copies, so
**C2 is untouched**.

In [ ]:
import shutil
import time

STAGE = P5 / 'stage'
STAGE.mkdir(parents=True, exist_ok=True)


def stage(src, name):
    """Copy a Drive path to VM-local disk once; return the local path."""
    if src is None or not src.exists():
        return src
    dst = STAGE / name
    if dst.exists():
        print(f'  {name:12s} already staged')
        return dst
    t0 = time.time()
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    files = [f for f in dst.rglob('*') if f.is_file()] if dst.is_dir() else [dst]
    mib = sum(f.stat().st_size for f in files) / 2**20
    print(f'  {name:12s} {len(files):5d} files, {mib:7.1f} MiB in {time.time() - t0:5.1f}s')
    return dst


print('staging to VM-local NVMe:')
L_CLIP = stage(CLIP_DIR, 'clip')
L_T2 = stage(T2, 'corpus')
L_FLOW = stage(FLOW_DIR, 'flow')
L_KNN = stage(KNN_CACHE, 'knn_cache.npz')

## 1. Preflight — **the trap from plan §6.1, checked before anything is spent**

`results.json` records checkpoints at `/content/p4/runs/...`, a **VM-local** path of a runtime that no
longer exists. phase_4's `sync_to_drive` *did* copy `checkpoint_last.pt` after every chunk, so they should
be on Drive — but "should" is what lesson **C10** exists to refuse. Count the files.

This cell also recovers **K**, the measured `kip_rec` D1 is judged against, from `metrics.jsonl` rather
than from a number typed into a notebook.

In [ ]:
import json
import statistics
import subprocess

ENV = {**os.environ, 'PYTHONPATH': str(REPO)}


def run(cmd, cwd=REPO, capture=False):
    """Run a child; make its failure legible instead of a bare CalledProcessError."""
    proc = subprocess.run([str(c) for c in cmd], cwd=str(cwd), env=ENV,
                          capture_output=capture, text=True)
    if proc.returncode:
        if capture:
            print(proc.stdout or '', proc.stderr or '', sep='\n')
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd)}'
                           + ('' if capture else '  -- scroll up for the child output'))
    if capture and proc.stdout:
        print(proc.stdout.rstrip())


def last_epoch_mean(run_dir, key):
    """Mean of `key` over the final epoch's batches -- the number the arm ended at."""
    path = run_dir / 'metrics.jsonl'
    if not path.exists():
        return None
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    if not rows:
        return None
    final = max(r['epoch'] for r in rows)
    values = [r[key] for r in rows if r['epoch'] == final and key in r]
    return statistics.fmean(values) if values else None


# --- flow cache: the target D1 measures -------------------------------------
flow_targets = sorted(FLOW_DIR.glob('*.npy'))
flow_stats_files = [p for p in flow_targets if p.name.endswith('.stats.npy')]
flow_e_o = [p for p in flow_targets if not p.name.endswith('.stats.npy')]
train_ids = [ln.strip() for ln in (T2 / 'train_ids.txt').read_text().splitlines() if ln.strip()] \
    if (T2 / 'train_ids.txt').exists() else []
print(f'flow cache  {FLOW_DIR}')
print(f'  e_O arrays      {len(flow_e_o):5d}')
print(f'  .stats.npy      {len(flow_stats_files):5d}')
print(f'  projection      {"OK" if (constants.FLOW_CACHE_DIR / constants.FLOW_PROJECTION_FILENAME).exists() else "MISSING"}')
if train_ids:
    print(f'  train ids       {len(train_ids):5d} (window ids; flow is keyed by SOURCE clip)')
assert flow_e_o and flow_stats_files, (
    'D1 needs BOTH {id}.npy and {id}.stats.npy under the flow cache. Run phase_4 section 3 '
    '(RUN_FLOW=True) first.')

# --- checkpoints: what D2 can be run against --------------------------------
CKPTS = {}
print(f'\ncheckpoints under {RUNS}')
for seed in SEEDS:
    for point in ('stage1', 'stage2_kip_on', 'stage2_kip_off'):
        ckpt = RUNS / f's{seed}' / point / 'checkpoint_last.pt'
        ok = ckpt.is_file() and ckpt.stat().st_size > 0
        if ok:
            CKPTS[(seed, point)] = ckpt
        size = f'{ckpt.stat().st_size / 2**20:8.1f} MiB' if ok else '   ABSENT'
        print(f'  s{seed} {point:15s} {size}')

# --- K: the measured kip_rec the D1 bars judge ------------------------------
K_PER_SEED = {s: last_epoch_mean(RUNS / f's{s}' / 'stage2_kip_on', 'kip_rec') for s in SEEDS}
K_PER_SEED = {s: v for s, v in K_PER_SEED.items() if v is not None}
assert K_PER_SEED, f'no stage2_kip_on metrics.jsonl under {RUNS} -- nothing to judge D1 against'
K = statistics.fmean(K_PER_SEED.values())
_s1 = [v for v in (last_epoch_mean(RUNS / f's{s}' / 'stage1', 'kip_rec') for s in SEEDS)
       if v is not None]
K_STAGE1 = statistics.fmean(_s1) if _s1 else None
print(f'\nK (mean final-epoch kip_rec, stage 2 KIP-on) = {K:.4f}   per seed '
      + ', '.join(f's{s}={v:.3f}' for s, v in K_PER_SEED.items()))
if K_STAGE1 is None:
    print('stage-1 plateau                              = n/a (no stage1 metrics.jsonl on Drive)')
else:
    print(f'stage-1 plateau (trunk frozen)               = {K_STAGE1:.4f}   '
          f'-> stage 2 bought {100 * (K_STAGE1 - K) / K_STAGE1:.0f} % of it by moving the trunk')

if not CKPTS:
    print('\n*** D2 CANNOT RUN: no checkpoint survived on Drive. Section 3 will stop. ***')

## 2. **D1** — what a measured `kip_rec` is worth

`core.tools.eda report --sections features --no-probe` streams the flow cache and reports the MSE three
reference predictors score on the same `e_O` the loss sees:

| symbol | predictor | reads |
|---|---|---|
| `Z` | all zeros | the loss at step 1 of an untrained PMG head |
| `V` | one global mean vector | **the baseline `kip_rec` must beat** |
| `W` | each item's own mean | an oracle that knows item identity and nothing else |

`--no-probe` because Gate D0 already answered the representation question at **0.6518**; this run is about
the flow cache only.

In [ ]:
D1_OUT = DIAG / 'd1_flow_target'
D1_JSON = D1_OUT / constants.EDA_REPORT_JSON_FILENAME

if D1_JSON.exists():
    print(f'reusing {D1_JSON}')
else:
    run([sys.executable, '-m', 'core.tools.eda', 'report',
         '--dataset', DATASET,
         '--data-dir', L_T2,
         '--clip-dir', L_CLIP,
         '--flow-dir', L_FLOW,
         '--output-dir', D1_OUT,
         '--sections', 'features',
         '--no-probe'], capture=True)

d1 = json.loads(D1_JSON.read_text(encoding='utf-8'))
flow = d1['features']['flow']
target = flow.get('target')
assert target, (
    "the report carries no 'target' block: the flow dir has .stats.npy but no {id}.npy, "
    'or this checkout predates the D1 extension of core/eda/features.py')

Z = target['mse_zero_predictor']
V = target['mse_global_mean_predictor']
W = target['mse_item_mean_predictor']
PRED = target['predicted_second_moment_from_raw']
print(f"items {target['items']:,} | frames {target['frames']:,} | dims {target['dims']}")
print(f'  Z  all-zeros predictor        {Z:10.4f}')
print(f'  V  global-mean predictor      {V:10.4f}   <- the baseline')
print(f'  W  item-mean oracle           {W:10.4f}')
print(f'  K  measured kip_rec           {K:10.4f}')
print(f"  between-item share of variance {target['between_item_share']:9.1%}")
print('\ntop raw stats by share of E[s^2] -- this is what sets the scale:')
for row in flow['raw_energy_share'][:5]:
    print(f"  {row['name']:16s} mean {row['mean']:10.3f}  std {row['std']:10.3f}  "
          f"share {row['energy_share']:7.2%}")

### 2.1 The D1 bars

Each bar prints its own condition and its own verdict. **Do not read past a HARD STOP.**

In [ ]:
def verdict(flag, ok_word, bad_word):
    return f'{ok_word}' if flag else f'{bad_word}'


R2_GLOBAL = 1.0 - K / V if V > 0 else float('nan')
R2_ITEM = 1.0 - K / W if W > 0 else float('nan')
RATIO = PRED / Z if Z > 0 else float('nan')

D1D_OK = D1D_RATIO_LO <= RATIO <= D1D_RATIO_HI
D1A_PASS = R2_GLOBAL >= D1A_R2_PASS
D1B_PASS = K < W
D1C_OVERWEIGHT = V >= D1C_OVERWEIGHT_MSE

# The band is not arbitrary. `predicted` is mean_j E[s_j^2], which equals the target's
# per-dim second moment only in EXPECTATION over the projection; the realized matrix is
# fixed, so its Gram deviates by ~1/sqrt(d_O) = 1/sqrt(256) ~ 6 %. [0.90, 1.10] is about
# 1.6 of those. A 0.95 is the estimator; a 0.3 is a broken cache.
print(f'D1-d  projection round-trip   predicted/Z = {RATIO:.4f}   '
      f'bar [{D1D_RATIO_LO}, {D1D_RATIO_HI}] (~1.6x the {1 / constants.FLOW_DIM ** 0.5:.1%} '
      f'Gram spread at d_O={constants.FLOW_DIM})   '
      f'{verdict(D1D_OK, "OK", "*** HARD STOP ***")}')
if not D1D_OK:
    raise SystemExit(
        f'The cache and the seeded projection disagree (predicted {PRED:.4f} vs measured {Z:.4f}).\n'
        'Every kip_rec ever measured on this cache is uninterpretable until that is explained.\n'
        'Do NOT re-derive the projection to clear this -- flow_projection.npz is the artifact of '
        'record (plan section 6, risk 4).')

print(f'D1-a  R2 vs global mean       1 - K/V     = {R2_GLOBAL:+.4f}   '
      f'bar >= {D1A_R2_PASS}          {verdict(D1A_PASS, "PASS", "STOP")}')
print(f'D1-b  beats the item oracle   K={K:.3f} vs W={W:.3f}   '
      f'(R2 vs W = {R2_ITEM:+.4f})   {verdict(D1B_PASS, "PASS", "*** REFUTES ***")}')
print(f'D1-c  target scale            V           = {V:.4f}   '
      f'bar >= {D1C_OVERWEIGHT_MSE}          '
      f'{verdict(D1C_OVERWEIGHT, "CONFIRMS overweight", "no overweight")}')
if D1C_OVERWEIGHT:
    print(f'      -> the weight that would equalize L_KIP_rec with an O(1) task loss is '
          f'1/V = {1.0 / V:.4f}  (DERIVED, not adopted -- plan section 5 option B)')

print('\nEDA verdicts:')
for v in d1['verdicts']:
    print(f"  [{v['level']}] {v['title']}")
    print(f"        {v['detail']}")
    print(f"     -> {v['action']}")

## 3. **D2** — who actually steers the trunk

`core.tools.grad_probe` computes the stage-2 loss once per batch, then takes a **separate `autograd.grad`
per weighted term** against four parameter groups, and reports at the temporal encoder:

* `rho = |g_KIP| / |g_task|` — how much more the trunk moves for KIP than for the anomaly objective;
* `cos(g_kip_rec, g_task)` — whether the two **disagree about the direction**, or merely about the size.

Read-only: no optimizer step, no checkpoint written. It re-sums its weighted terms and **raises** if they
do not reproduce `compute_losses`'s own `total`, so a probe that runs is a probe whose split is the one
being descended.

Two points per seed, so the trajectory is visible: **`stage1`** (what stage 2 started from, evaluated under
the stage-2 objective) and **`stage2_kip_on`** (the state that produced the AUC above).

> `--config` is the **kip_on** config at both points, deliberately. `--batch-size` is left at the config's
> own 64 so the geometry is the one that trained; drop it to 16 only if the probe OOMs.
> Batches come from the epoch-0 permutation — the end-point model has seen them, which is fine for a
> gradient-geometry probe and is **not** a generalization measurement.

In [ ]:
D2_POINTS = [(s, p) for s in SEEDS for p in ('stage1', 'stage2_kip_on')]
NUM_BATCHES = 8

if not CKPTS:
    raise SystemExit('No checkpoint on Drive -- D2 cannot run. See section 1 and plan section 6 risk 1.')

D2 = {}
for seed, point in D2_POINTS:
    ckpt = CKPTS.get((seed, point))
    if ckpt is None:
        print(f's{seed} {point:15s} SKIPPED -- no checkpoint')
        continue
    out = DIAG / 'd2_grad' / f's{seed}_{point}'
    payload_path = out / 'grad_probe.json'
    if not payload_path.exists():
        print(f's{seed} {point:15s} probing {NUM_BATCHES} batches ...')
        run([sys.executable, '-m', 'core.tools.grad_probe',
             '--config', RUNS / f's{seed}' / 'stage2_kip_on' / 'config.yaml',
             '--data-dir', L_T2, '--clip-dir', L_CLIP, '--flow-dir', L_FLOW,
             '--knn-cache', L_KNN,
             '--checkpoint', ckpt,
             '--output-dir', out,
             '--num-batches', NUM_BATCHES], capture=True)
    D2[(seed, point)] = json.loads(payload_path.read_text(encoding='utf-8'))
print(f'\n{len(D2)} probes on disk under {DIAG / "d2_grad"}')

In [ ]:
def spread(payload, getter):
    """Mean and sample sd over the probe's per-batch records -- report both (n=8)."""
    values = [getter(r) for r in payload['records'] if 'trunk' in r]
    values = [v for v in values if v is not None]
    if not values:
        return float('nan'), float('nan')
    sd = statistics.stdev(values) if len(values) > 1 else 0.0
    return statistics.fmean(values), sd


print(f'{"point":22s} {"rho(all KIP)":>16s} {"rho(kip_rec)":>16s} {"cos(kip_rec,task)":>20s}')
print('-' * 78)
ROWS = {}
for (seed, point), payload in sorted(D2.items()):
    s = payload['summary']
    rho_all = s['trunk_kip_total_over_task']
    rho_rec, rho_rec_sd = spread(payload, lambda r: r['trunk']['kip_over_task'].get('kip_rec'))
    cos_rec, cos_rec_sd = spread(payload, lambda r: r['trunk']['cosine_with_task'].get('kip_rec'))
    ROWS[(seed, point)] = (rho_all, rho_rec, rho_rec_sd, cos_rec, cos_rec_sd)
    print(f's{seed} {point:15s} {rho_all:16.3f} {rho_rec:9.3f}+-{rho_rec_sd:<5.3f} '
          f'{cos_rec:12.3f}+-{cos_rec_sd:<5.3f}')

print('\nper-term, at the stage-2 end point (mean over seeds):')
END = {k: v for k, v in D2.items() if k[1] == 'stage2_kip_on'}
if END:
    keys = sorted({k for p in END.values() for k in p['summary']['trunk_kip_over_task']})
    for key in keys:
        r = statistics.fmean(p['summary']['trunk_kip_over_task'][key] for p in END.values())
        c = statistics.fmean(p['summary']['trunk_cosine_with_task'][key] for p in END.values())
        print(f'  {key:12s} rho {r:8.3f}   cos {c:+.3f}')

### 3.1 The D2 bars

In [ ]:
assert END, 'no stage-2 end-point probe -- the D2 bars are defined at that point'
RHO = statistics.fmean(p['summary']['trunk_kip_total_over_task'] for p in END.values())
COS = statistics.fmean(p['summary']['trunk_cosine_with_task']['kip_rec'] for p in END.values())
RHO_START = (statistics.fmean(p['summary']['trunk_kip_total_over_task']
                              for k, p in D2.items() if k[1] == 'stage1')
             if any(k[1] == 'stage1' for k in D2) else float('nan'))

if RHO >= D2A_CONFIRM:
    D2A = 'CONFIRM'
elif RHO < D2A_REFUTE:
    D2A = 'REFUTE'
else:
    D2A = 'PARTIAL'

if COS <= -D2B_COS:
    D2B = 'conflict'
elif abs(COS) < D2B_COS:
    D2B = 'orthogonal'
else:
    D2B = 'aligned'

print(f'D2-a  trunk gradient share    rho = {RHO:.3f}   '
      f'bar >= {D2A_CONFIRM} CONFIRM / < {D2A_REFUTE} REFUTE   -> **{D2A}**')
print(f'D2-b  direction               cos = {COS:+.3f}  '
      f'bar <= -{D2B_COS} conflict / |cos| < {D2B_COS} orthogonal / >= +{D2B_COS} aligned   -> **{D2B}**')
print(f'D2-c  trajectory              rho stage1 = {RHO_START:.3f} -> stage2 end = {RHO:.3f}   '
      '(reported, no bar)')

print({
    'CONFIRM': '\n  The trunk is majority-steered by KIP. The +26 % task loss and the -0.0129 '
               'micro have a mechanism.',
    'REFUTE':  '\n  The loss share does NOT become a gradient share. lambda_rec is the WRONG lever; '
               'the AUC drop needs another cause (start with C24, the fixed ~50 % shift).',
    'PARTIAL': '\n  Mechanism present but not dominant. No arm fires automatically -- decide with '
               'the user.',
}[D2A])
print({
    'conflict':   '  The two gradients disagree on direction: lowering lambda_rec TRADES one '
                  'objective for the other. The detach arm is the decisive control.',
    'orthogonal': '  KIP spends trunk capacity without fighting the task. Normalization plus a '
                  'derived weight is the coherent repair.',
    'aligned':    '  KIP pulls the trunk the same way the task does -- gradient conflict is NOT '
                  'the mechanism. Look at the shift (C24) and at L_kin instead.',
}[D2B])

## 4. The decision table (plan §4) — which row fires

This prints the row; **it does not run the arm.** Every option in plan §5 stays unauthorized until the user
picks one.

In [ ]:
# Section 3 raises SystemExit when no checkpoint survived, so D2 may not have run.
D2A = globals().get('D2A')
D2B = globals().get('D2B')

if not D1D_OK:
    ROW, NEXT = 'HARD STOP', 'D1-d failed: explain the cache/projection mismatch before anything else.'
elif not D1A_PASS:
    ROW = 'D1-a STOP'
    NEXT = ('The flow target is noise at this capacity. Normalizing it changes the number, not the '
            'situation -> plan section 5 option D (write the negative result). lambda_rec is not the story.')
elif not D1B_PASS:
    ROW = 'D1-b REFUTE'
    NEXT = ('PMG fits item identity, not motion -- it does not beat an oracle that knows only which '
            'window this is. KIP\'s premise on T2 is refuted independently of any weighting -> option D.')
elif D2A is None:
    ROW = 'D1 only'
    NEXT = ('D1 passed its bars but D2 did not run (no checkpoint on Drive). The trunk-capture '
            'question is still open -- re-run section 3 once a checkpoint exists, or re-train one '
            'arm to recover the end point. Do NOT pick a repair off D1 alone.')
elif D2A == 'REFUTE':
    ROW = 'D2-a REFUTE'
    NEXT = ('Loss share is not gradient share. Stop tuning lambda_rec; re-open the AUC drop against '
            'C24 (the fixed ~50 % shift) and L_kin.')
elif D2A == 'CONFIRM' and D2B == 'conflict':
    ROW = 'capture + conflict'
    NEXT = ('Run option C (detach v^t before PMG) as the decisive control FIRST, then option A. '
            'A weight sweep alone cannot separate capture from conflict.')
elif D2A == 'CONFIRM' and D2B in ('orthogonal', 'aligned'):
    ROW = f'capture + {D2B}'
    NEXT = (f'Option A: z-score e_O from train-split statistics (NEW cache version -- C2 -- never '
            f'overwrite flow/v1), with lambda_rec re-derived as 1/V = {1.0 / V:.4f}, not swept.')
else:
    ROW = 'partial'
    NEXT = 'Report both numbers and decide with the user. No arm fires automatically.'

print(f'decision row: **{ROW}**\n')
print(f'next: {NEXT}\n')
print('Reminder, whatever the row says:')
print('  - C24: every KIP-on arm on `main` is a fixed ~50 % channel shift. Never write "motion-gated".')
print('  - lesson 14: if lambda_rec is adopted it must be DERIVED from D1-c, not searched against a delta.')
print('  - C2: changing the flow target invalidates every KIP-on number already measured on flow/v1.')

## 5. Record the run (**C17**)

A run that does not record what it was is a run that has to be re-done. `commit: UNKNOWN` is expected here —
the Drive copy of the repo is uploaded, not cloned — so the manifest records a **sha256 of `core/**/*.py`**
instead, which answers the same question.

In [ ]:
import hashlib
import platform

def core_sha256(repo):
    """One digest over every core/**/*.py, sorted -- what `git rev-parse` would answer."""
    digest = hashlib.sha256()
    for path in sorted((repo / 'core').rglob('*.py')):
        digest.update(str(path.relative_to(repo)).encode())
        digest.update(path.read_bytes())
    return digest.hexdigest()


try:
    commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=str(REPO),
                            capture_output=True, text=True).stdout.strip() or 'UNKNOWN'
except Exception:
    commit = 'UNKNOWN'

manifest = {
    'notebook': 'colab/DADA2000Origin/diag_kip_loss_scale.ipynb',
    'plan': '.project/plans/katvad-kip-loss-scale-diagnosis.md',
    'dataset': DATASET,
    'commit': commit,
    'core_sha256': core_sha256(REPO),
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'platform': platform.platform(),
    'explains': 'outputs/v1/DADA2000_orig_phase4 (3 seeds, paired kip on/off)',
    'bars': {
        'D1A_R2_PASS': D1A_R2_PASS, 'D1C_OVERWEIGHT_MSE': D1C_OVERWEIGHT_MSE,
        'D1D_RATIO': [D1D_RATIO_LO, D1D_RATIO_HI],
        'D2A_CONFIRM': D2A_CONFIRM, 'D2A_REFUTE': D2A_REFUTE, 'D2B_COS': D2B_COS,
    },
    'd1': {
        'K_measured_kip_rec': K, 'K_per_seed': K_PER_SEED, 'K_stage1_plateau': K_STAGE1,
        'Z_zero': Z, 'V_global_mean': V, 'W_item_mean': W,
        'R2_global': R2_GLOBAL, 'R2_item': R2_ITEM,
        'projection_ratio': RATIO,
        'between_item_share': target['between_item_share'],
        'top_raw_stat': flow['raw_energy_share'][0],
        'lambda_rec_equalizing': 1.0 / V if V > 0 else None,
        'verdicts': {'D1a': D1A_PASS, 'D1b': D1B_PASS, 'D1c': D1C_OVERWEIGHT, 'D1d': D1D_OK},
    },
    'd2': {
        'num_batches': NUM_BATCHES,
        'points': {f's{s}_{p}': dict(zip(
            ('rho_all', 'rho_kip_rec', 'rho_kip_rec_sd', 'cos_kip_rec', 'cos_kip_rec_sd'), v))
            for (s, p), v in ROWS.items()},
        'rho_end': RHO, 'rho_stage1': RHO_START, 'cos_end': COS,
        'verdicts': {'D2a': D2A, 'D2b': D2B},
    },
    'decision_row': ROW,
    'next': NEXT,
}
path = DIAG / 'diag_manifest.json'
path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'wrote {path}')
print(json.dumps({k: manifest[k] for k in ('decision_row', 'next')}, indent=2, ensure_ascii=False))

---

## What to bring back

Copy these off the VM before the runtime is recycled — `outputs/` is gitignored, so **these files are the
only durable record**:

* `$KATVAD_OUTPUT_ROOT/DADA2000_orig_diag_kip_loss_scale/diag_manifest.json` — the verdicts and the row;
* `.../d1_flow_target/eda_report.{json,md}` — §4.3 / §4.3.1, the baselines and the energy table;
* `.../d2_grad/s*/grad_probe.{json,md}` — every per-batch record, not just the means.

Then report **K, V, W, rho, cos and the decision row**. The row names the next arm; nothing in plan §5 is
authorized until it is chosen.